# 2026si05077 - deep neural networks programming assignment

# a single neuron for regression, and a single neuron for classification

this notebook has two parts.

part a builds a linear neural network with one output neuron that predicts a continuous
number (regression).

part b builds a linear neural network with one output neuron that
predicts one of two classes (binary classification).

both parts are written from scratch using only numpy for the maths. pandas is used to
load and organise the data, matplotlib is used for plots, and sklearn is only used for the
allowed helper tools, train test split and standard scaler, never for the model itself.

both parts follow the same idea at heart, a single neuron takes some input numbers,
multiplies each one by a weight, adds them all up together with a bias term, and produces
one output number. the only difference between the two parts is what we do with that
output number afterwards, and how we measure the error.

In [1]:
# import the libraries we need for both parts of this notebook
# numpy handles all the array math, this is the only library we use to build the model itself
# pandas loads and organises our tables of data
# matplotlib draws all of our plots
# xarray is only used to open the .nc4 grid file, it does not do any modelling
# from sklearn we only use the allowed helper tools, never an actual model
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
)

# a fixed seed means we get the exact same random numbers every time we run this notebook
# this makes our results repeatable, so we can compare runs fairly
np.random.seed(42)

# this is where all output plots get saved as image files, so they can be attached as
# screenshots when this assignment is submitted
OUTPUT_DIR = Path("outputs")
(OUTPUT_DIR / "regression").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "classification").mkdir(parents=True, exist_ok=True)

ModuleNotFoundError: No module named 'numpy'

---

# part a: linear neural network for regression (5 marks)

## problem statement

instead of a kaggle dataset, we use a real scientific dataset already available to us, an
ocean model grid file in netcdf4 format (`114549.nc4`). a `.nc4` file is just a
self describing container for large scientific data, here it stores the longitude and
latitude of every cell in an ocean grid, together with a land sea mask and the ocean depth
at each cell.

our regression question is simple: **can a single neuron predict how deep the ocean is at
a location, just from its longitude and latitude?**

- input `x`: longitude and latitude of a grid cell (two numbers)
- output `y`: seafloor depth in meters (one continuous number)

we only keep sea cells, land cells do not have a meaningful depth value.

## model description

- output equation: `y_hat = w . x + b` (a weighted sum of the inputs plus a bias, this is
  exactly what one neuron computes, when there is more than one input, `w` and `x` are
  simply lists of numbers instead of single numbers, but the idea does not change)
- loss function: mean squared error, `L = (1/n) * sum((y_i - y_hat_i)^2)`
- optimisation method: batch gradient descent, meaning we look at the whole training set
  before making a single update to `w` and `b`

In [ ]:
# open the .nc4 grid file
# xarray reads the file and shows us every variable it contains, along with its shape
file_path = "data/regression/114549.nc4"
ds = xr.open_dataset(file_path)
print(ds)

In [ ]:
# pull out only the four variables we actually need
nav_lon = ds["nav_lon"].values          # shape (y, x), longitude of every grid cell
nav_lat = ds["nav_lat"].values          # shape (y, x), latitude of every grid cell
tmaskutil = ds["tmaskutil"].values[0]   # shape (y, x), 1 means sea, 0 means land
mbathy = ds["mbathy"].values[0]         # shape (y, x), how many wet vertical levels each column has
gdepw_1d = ds["gdepw_1d"].values[0]     # shape (z,), depth in meters at each vertical level

print("grid shape:", nav_lon.shape)
print("fraction of the grid that is sea:", round(tmaskutil.mean(), 3))

In [ ]:
# flatten the 2d grid into a plain table, one row per grid cell
lon_flat = nav_lon.ravel()
lat_flat = nav_lat.ravel()
mask_flat = tmaskutil.ravel()
mbathy_flat = mbathy.ravel()

# keep sea cells only, land cells have no meaningful ocean depth
is_sea = mask_flat == 1

# turn the number of wet levels into an actual depth in meters using the lookup table
depth_m = gdepw_1d[mbathy_flat[is_sea]]

reg_df = pd.DataFrame({
    "longitude": lon_flat[is_sea],
    "latitude": lat_flat[is_sea],
    "depth_m": depth_m,
})

# a few coastal cells are marked as sea but have zero wet levels, we drop those noisy rows
reg_df = reg_df[reg_df["depth_m"] > 0].reset_index(drop=True)

print("number of usable sea grid cells:", len(reg_df))
reg_df.head()

In [ ]:
# look at the data before modelling
# this map colours every grid cell by how deep the ocean is at that point
plt.figure(figsize=(10, 5))
scatter = plt.scatter(reg_df["longitude"], reg_df["latitude"], c=reg_df["depth_m"], cmap="viridis", s=1)
plt.colorbar(scatter, label="seafloor depth (m)")
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.title("ocean depth across the model grid")
plt.savefig(OUTPUT_DIR / "regression" / "partA_01_depth_map.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# this dataset has over sixty thousand sea cells, we take a random sample so training stays
# fast, while still keeping enough points to see the pattern clearly
SAMPLE_SIZE = 10000
reg_sample = reg_df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print("sample size used for training:", len(reg_sample))

In [ ]:
# group task 1 (part a, step 1 of 7): prepare the data before we initialise anything
# x is our input matrix (longitude, latitude), y is the target we want to predict (depth)
X = reg_sample[["longitude", "latitude"]].values
y = reg_sample["depth_m"].values

# split into eighty percent training data and twenty percent testing data
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scale the inputs so both features sit on a similar range, this helps gradient descent
# converge smoothly, we fit the scaler on training data only, then apply it to the test data
# so no information from the test set leaks into training
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print("training set shape:", X_train.shape)
print("testing set shape:", X_test.shape)

## group tasks

the seven steps below match the assignment brief exactly:
1. initialise weight `w` and bias `b`
2. forward pass to compute predictions
3. calculate mse loss
4. derive gradients of the loss with respect to `w` and `b`
5. update the parameters using gradient descent
6. train for a fixed number of epochs
7. display the final parameters and prediction results

In [ ]:
class LinearNeuronRegressor:
    # a single linear neuron trained with batch gradient descent
    # this class is deliberately built using nothing but numpy, no model comes from any library

    def __init__(self, learning_rate=0.1, epochs=500):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.w = None
        self.b = None
        self.loss_history = []

    def _mse_loss(self, y_true, y_pred):
        # group task 3: calculate the mean squared error loss
        # this is the average of the squared difference between the true and predicted values
        # squaring the error means big mistakes are punished much more than small ones
        return np.mean((y_true - y_pred) ** 2)

    def fit(self, X, y):
        num_samples, num_features = X.shape

        # group task 1: initialise weight w and bias b
        # we start every weight and the bias at zero, gradient descent will move them from here
        self.w = np.zeros(num_features)
        self.b = 0.0

        for epoch in range(self.epochs):
            # group task 2: forward pass, compute the prediction y_hat = w . x + b for every
            # training example at once, this is a batch forward pass
            y_hat = np.dot(X, self.w) + self.b

            # group task 3: calculate the mse loss for this epoch
            loss = self._mse_loss(y, y_hat)
            self.loss_history.append(loss)

            # group task 4: derive the gradients of the loss with respect to w and b
            # these formulas come directly from taking the derivative of the mse loss
            error = y_hat - y
            grad_w = (2 / num_samples) * np.dot(X.T, error)
            grad_b = (2 / num_samples) * np.sum(error)

            # group task 5: update the parameters, moving them a small step in the
            # direction that makes the loss smaller
            self.w -= self.learning_rate * grad_w
            self.b -= self.learning_rate * grad_b

            if (epoch + 1) % 50 == 0 or epoch == 0:
                print(f"epoch {epoch + 1}/{self.epochs}, mse loss: {loss:.2f}")

    def predict(self, X):
        return np.dot(X, self.w) + self.b

In [ ]:
# group task 6: train the model for a fixed number of epochs
# learning rate controls how big each step is, epochs controls how many times we repeat the
# whole forward pass, loss calculation, gradient and update cycle
regressor = LinearNeuronRegressor(learning_rate=0.1, epochs=500)
regressor.fit(X_train, y_train)

In [ ]:
# plot the loss curve, a curve that drops quickly and then flattens means the model
# has converged, meaning further training will not improve it much more
plt.figure(figsize=(8, 5))
plt.plot(regressor.loss_history)
plt.xlabel("epoch")
plt.ylabel("training loss (mean squared error)")
plt.title("part a: loss curve, batch gradient descent")
plt.grid(True)
plt.savefig(OUTPUT_DIR / "regression" / "partA_02_loss_curve.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# group task 7: display the final parameters and prediction results
y_pred_test = regressor.predict(X_test)

mse = mean_squared_error(y_test, y_pred_test)
mae = mean_absolute_error(y_test, y_pred_test)
r2 = r2_score(y_test, y_pred_test)

print("final learned weight for longitude:", regressor.w[0])
print("final learned weight for latitude:", regressor.w[1])
print("final learned bias:", regressor.b)
print()
print(f"test mean squared error: {mse:.2f}")
print(f"test mean absolute error: {mae:.2f} meters")
print(f"test r squared: {r2:.4f}")

In [ ]:
# a predicted vs actual plot, a perfect model would place every point exactly on the
# diagonal red line
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred_test, s=3, alpha=0.4)
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
plt.plot(lims, lims, color="red", linestyle="--", label="perfect prediction")
plt.xlabel("actual depth (m)")
plt.ylabel("predicted depth (m)")
plt.title("part a: predicted vs actual seafloor depth")
plt.legend()
plt.savefig(OUTPUT_DIR / "regression" / "partA_03_predicted_vs_actual.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# show a handful of individual predictions next to their true values, this makes the
# final results easy to read at a glance
results_preview = pd.DataFrame({
    "actual_depth_m": y_test[:10],
    "predicted_depth_m": np.round(y_pred_test[:10], 2),
})
results_preview

## part a observations

a single linear neuron with two inputs can only draw a flat plane through the data, but the
real ocean floor is bumpy, full of trenches and shelves. this is why the r squared score is
low, the model correctly learns the broad, large scale trend of depth across the globe, but
it cannot capture small local features. this is expected and reasonable for the simplest
possible linear model, and matches exactly what the assignment asks for, a single neuron
with a linear output.

---

# part b: linear neural network for classification (5 marks)

## problem statement

instead of a kaggle dataset, we use motion sensor recordings collected from an arduino
board performing different gestures (circle, idle, left-right, up-down). each recording is
a short time series of accelerometer and gyroscope readings.

our classification question is simple: **can a single neuron tell whether the device is
moving or standing still, just from a short window of its motion sensor readings?**

- input `x`: summary statistics (mean, standard deviation, min, max, energy) computed over
  a one second window of the six sensor channels, giving 30 input numbers per window
- output `y`: 0 if the device was idle, 1 if the device was performing any gesture

## model description

- net input: `z = w . x + b`
- activation function, sigmoid: `sigmoid(z) = 1 / (1 + e^(-z))`, this squashes the net
  input into a probability between 0 and 1
- decision rule: predict class 1 if `sigmoid(z) >= 0.5`, otherwise predict class 0
- loss function: binary cross entropy

In [ ]:
# load every recording from the training and testing folders
# each file name looks like circle.xxxx.csv, the part before the first dot is the gesture
SENSOR_COLUMNS = ["accX", "accY", "accZ", "gyrX", "gyrY", "gyrZ"]
KNOWN_LABELS = {"circle", "idle", "left-right", "up-down"}

TRAIN_DIR = Path("data/classification/tinyml_demo-export/training-csv")
TEST_DIR = Path("data/classification/tinyml_demo-export/testing-csv")


def load_gesture_files(folder):
    data_list, label_list = [], []
    for csv_path in sorted(folder.glob("*.csv")):
        label = csv_path.name.split(".")[0]
        if label not in KNOWN_LABELS:
            # a couple of stray files are not real gesture recordings, we skip those
            continue
        df = pd.read_csv(csv_path)
        data_list.append(df[SENSOR_COLUMNS].values)
        label_list.append(label)
    return data_list, label_list


train_recordings, train_gesture_labels = load_gesture_files(TRAIN_DIR)
test_recordings, test_gesture_labels = load_gesture_files(TEST_DIR)

print("number of training recordings:", len(train_recordings))
print("number of testing recordings:", len(test_recordings))

In [ ]:
# cut every recording into overlapping one second windows
# each sample is collected roughly every 16 milliseconds, so 62 rows is close to one second
WINDOW_SIZE = 62
STEP_SIZE = 31   # fifty percent overlap gives us more windows to train on


def make_windows(recordings, labels, window_size=WINDOW_SIZE, step_size=STEP_SIZE):
    windows, window_labels = [], []
    for recording, label in zip(recordings, labels):
        for start in range(0, len(recording) - window_size, step_size):
            windows.append(recording[start:start + window_size])
            window_labels.append(label)
    return windows, window_labels


train_windows, train_window_labels = make_windows(train_recordings, train_gesture_labels)
test_windows, test_window_labels = make_windows(test_recordings, test_gesture_labels)

print("number of training windows:", len(train_windows))
print("number of testing windows:", len(test_windows))

In [ ]:
# turn every window into one row of input features
def extract_features(window):
    # mean is the average value, std is how much the values wiggle, min and max are the
    # extremes reached, energy shows how strong the overall signal is in that window
    feats = {}
    for i, name in enumerate(SENSOR_COLUMNS):
        values = window[:, i]
        feats[f"{name}_mean"] = values.mean()
        feats[f"{name}_std"] = values.std()
        feats[f"{name}_min"] = values.min()
        feats[f"{name}_max"] = values.max()
        feats[f"{name}_energy"] = np.sum(values ** 2) / len(values)
    return feats


def build_feature_table(windows, labels):
    rows = []
    for window, label in zip(windows, labels):
        row = extract_features(window)
        row["gesture"] = label
        row["is_moving"] = 0 if label == "idle" else 1
        rows.append(row)
    return pd.DataFrame(rows)


clf_train_df = build_feature_table(train_windows, train_window_labels)
clf_test_df = build_feature_table(test_windows, test_window_labels)

clf_train_df.head()

In [ ]:
# check the class balance, this tells us if one class has a lot more examples than the other
print("training label counts:")
print(clf_train_df["is_moving"].value_counts())
print()
print("testing label counts:")
print(clf_test_df["is_moving"].value_counts())

In [ ]:
# separate features from labels, then scale the features
feature_columns = [c for c in clf_train_df.columns if c not in ("gesture", "is_moving")]

X_train_clf_raw = clf_train_df[feature_columns].values
y_train_clf = clf_train_df["is_moving"].values

X_test_clf_raw = clf_test_df[feature_columns].values
y_test_clf = clf_test_df["is_moving"].values

# fit the scaler on training data only, then apply the same scaling to the test data
clf_scaler = StandardScaler()
X_train_clf = clf_scaler.fit_transform(X_train_clf_raw)
X_test_clf = clf_scaler.transform(X_test_clf_raw)

print("training feature matrix shape:", X_train_clf.shape)
print("testing feature matrix shape:", X_test_clf.shape)

## group tasks

the six steps below match the assignment brief exactly:
1. initialise weights and bias
2. compute the net input and the activated output
3. calculate the loss for classification
4. update the parameters using gradient descent
5. apply the threshold to obtain class labels
6. report predicted versus actual classes

In [ ]:
def sigmoid(z):
    # the sigmoid activation squashes any number into a probability between 0 and 1
    # we clip z first so we never compute exp of a huge number, which would overflow
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

In [ ]:
class LinearNeuronClassifier:
    # a single neuron with a sigmoid activation, this is exactly logistic regression
    # trained with stochastic gradient descent, updating after every single example

    def __init__(self, learning_rate=0.01, epochs=50):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.w = None
        self.b = None
        self.loss_history = []

    def _binary_cross_entropy(self, y_true, y_pred):
        # group task 3: calculate the loss for classification
        # this loss punishes the model heavily when it is confident and wrong
        # a tiny epsilon is added so we never take the log of zero
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

    def fit(self, X, y):
        num_samples, num_features = X.shape

        # group task 1: initialise weights and bias, we start at zero for every one
        self.w = np.zeros(num_features)
        self.b = 0.0

        for epoch in range(self.epochs):
            # shuffle the order of the examples every epoch, so the model does not
            # accidentally learn the order the data happens to be stored in
            indices = np.random.permutation(num_samples)

            for i in indices:
                xi = X[i]
                yi = y[i]

                # group task 2: compute the net input z and the activated output
                z = np.dot(xi, self.w) + self.b
                y_hat = sigmoid(z)

                # this is the gradient of the binary cross entropy loss for one example
                # group task 4: update the parameters using this gradient
                error = y_hat - yi
                self.w -= self.learning_rate * error * xi
                self.b -= self.learning_rate * error

            # after each full pass through the data, check the loss on the whole training set
            full_z = np.dot(X, self.w) + self.b
            full_y_hat = sigmoid(full_z)
            epoch_loss = self._binary_cross_entropy(y, full_y_hat)
            self.loss_history.append(epoch_loss)

            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(f"epoch {epoch + 1}/{self.epochs}, loss: {epoch_loss:.4f}")

    def predict_proba(self, X):
        # this returns the raw probability of belonging to class 1
        return sigmoid(np.dot(X, self.w) + self.b)

    def predict(self, X, threshold=0.5):
        # group task 5: apply the threshold to obtain the final 0 or 1 class labels
        probabilities = self.predict_proba(X)
        return (probabilities >= threshold).astype(int)

In [ ]:
# train the classifier
classifier = LinearNeuronClassifier(learning_rate=0.01, epochs=50)
classifier.fit(X_train_clf, y_train_clf)

In [ ]:
# plot the loss curve, a steadily falling curve that flattens out shows the model is
# learning in a stable way
plt.figure(figsize=(8, 5))
plt.plot(classifier.loss_history)
plt.xlabel("epoch")
plt.ylabel("training loss (binary cross entropy)")
plt.title("part b: loss curve, stochastic gradient descent")
plt.grid(True)
plt.savefig(OUTPUT_DIR / "classification" / "partB_01_loss_curve.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# group task 6: report predicted versus actual classes
y_pred_clf = classifier.predict(X_test_clf)

accuracy = accuracy_score(y_test_clf, y_pred_clf)
precision = precision_score(y_test_clf, y_pred_clf)
recall = recall_score(y_test_clf, y_pred_clf)
f1 = f1_score(y_test_clf, y_pred_clf)

print(f"accuracy: {accuracy:.4f}")
print(f"precision: {precision:.4f}")
print(f"recall: {recall:.4f}")
print(f"f1 score: {f1:.4f}")

In [ ]:
# a confusion matrix shows exactly what the model got right and wrong
# rows are the true labels, columns are the predicted labels
cm = confusion_matrix(y_test_clf, y_pred_clf)

plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap="Blues")
plt.title("part b: confusion matrix")
plt.xlabel("predicted label")
plt.ylabel("true label")
plt.xticks([0, 1], ["idle (0)", "moving (1)"])
plt.yticks([0, 1], ["idle (0)", "moving (1)"])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center", color="black")
plt.colorbar()
plt.savefig(OUTPUT_DIR / "classification" / "partB_02_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# show a handful of individual predictions next to their true labels
report_preview = pd.DataFrame({
    "actual_class": y_test_clf[:10],
    "predicted_class": y_pred_clf[:10],
    "predicted_probability": np.round(classifier.predict_proba(X_test_clf)[:10], 3),
})
report_preview

## part b observations

the single neuron with a sigmoid activation separates moving from idle windows very well,
because moving windows have much larger swings in acceleration and rotation than idle
ones, a pattern a single linear neuron can pick up easily. the confusion matrix and the
accuracy, precision, recall and f1 scores together show exactly how reliable the model is,
not just a single number.

## overall conclusion

both parts show the same core building block, a single neuron computing a weighted sum of
its inputs plus a bias. in part a, that raw sum is the final answer, so we use a squared
error loss to measure how far off a continuous prediction is. in part b, that raw sum is
passed through a sigmoid to turn it into a probability, so we use binary cross entropy to
measure how far off a predicted probability is from the true class. everything else, the
forward pass, the gradients, and the parameter update rule, follows the exact same pattern
in both cases.